# House Price Prediction — Educational Walkthrough

This notebook reuses the project's Python modules to walk through each step:
1. **Data loading** — Read the Ames Housing CSV
2. **Feature engineering** — Build TotalBathrooms and select our three predictors
3. **Exploratory data analysis** — Quick visual summary
4. **Train/validation split** — 80/20 split
5. **Model training** — Linear regression via scikit-learn Pipeline
6. **Evaluation** — MAE, RMSE, R², and plots
7. **Prediction** — Use the trained model on new data

> **Note:** This notebook calls functions from `src/features.py` and `src/train.py`
> rather than duplicating the pipeline.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Ensure project root is on the Python path
if Path('.').resolve().name == 'notebooks':
    PROJECT_ROOT = Path('.').resolve().parent
else:
    PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import build_features, validate_dataframe, FEATURE_COLS, TARGET_COL
from src.train import make_feature_pipeline, evaluate, save_validation_plots

from sklearn.model_selection import train_test_split

print('All imports successful.')

## 1. Load the Dataset

Make sure `data/train.csv` exists in the project directory.

In [ ]:
csv_path = PROJECT_ROOT / 'data' / 'train.csv'

if not csv_path.exists():
    print(f'Dataset not found at {csv_path}')
    print('Download from: https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data')
    print('Place train.csv inside the data/ directory and re-run this notebook.')
else:
    df = pd.read_csv(csv_path)
    validate_dataframe(df, role='training')
    print(f'Loaded {len(df)} rows and {len(df.columns)} columns.')
    df.head()

## 2. Data Description

Key columns:
- **SalePrice** — Target variable (what we want to predict)
- **GrLivArea** — Above-ground living area in square feet
- **BedroomAbvGr** — Number of bedrooms above ground
- **FullBath / HalfBath / BsmtFullBath / BsmtHalfBath** — Bathroom components

See `data/data_description.txt` for the full list of 79 attributes.

In [ ]:
# Quick summary of our target and raw predictors
print(df[['GrLivArea', 'BedroomAbvGr', 'SalePrice']].describe())

## 3. Feature Engineering

We build `TotalBathrooms` from four columns and select our three final predictors.

In [ ]:
features = build_features(df)
print('Feature columns:', list(features.columns))
features.head(10)

## 4. Exploratory Data Analysis

Visualise relationships between each feature and the sale price.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, FEATURE_COLS):
    ax.scatter(features[col], df[TARGET_COL], alpha=0.4, edgecolors='k', linewidths=0.3)
    ax.set_xlabel(col)
    ax.set_ylabel('SalePrice ($)')
    ax.set_title(f'{col} vs SalePrice')
fig.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
corr_df = features.copy()
corr_df[TARGET_COL] = df[TARGET_COL]
sns.heatmap(corr_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

## 5. Train / Validation Split

We use an 80/20 split with `random_state=42` for reproducibility.

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
print(f'Training: {len(train_df)} rows | Validation: {len(val_df)} rows')

## 6. Model Training

We use a scikit-learn `Pipeline`:
1. `FunctionTransformer` — builds the three features
2. `SimpleImputer(strategy='median')` — fills missing values
3. `LinearRegression` — the model

The pipeline is fitted **only** on the training split.

In [ ]:
pipeline = make_feature_pipeline()
pipeline.fit(train_df, train_df[TARGET_COL])
print('Pipeline fitted on training data.')

# View the learned coefficients
model = pipeline.named_steps['model']
print(f'\nIntercept: ${model.intercept_:,.2f}')
for name, coef in zip(FEATURE_COLS, model.coef_):
    print(f'  {name:20s}: ${coef:,.2f}')
print('\nCoefficients describe associations, not causal effects.')

## 7. Evaluation

We evaluate on the **validation** split (data the model has never seen).

In [ ]:
val_pred = pipeline.predict(val_df)
metrics = evaluate(val_df[TARGET_COL].values, val_pred)

print('--- Validation Metrics ---')
print(f'  MAE : ${metrics["MAE"]:,.2f}')
print(f'  RMSE: ${metrics["RMSE"]:,.2f}')
print(f'  R²  : {metrics["R2"]:.4f}')
print()
print('MAE = average dollar error.')
print('RMSE = average error penalising large mistakes more.')
print('R² = fraction of price variance explained (1.0 = perfect, 0.0 = no better than mean).')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Actual vs Predicted
axes[0].scatter(val_pred, val_df[TARGET_COL].values, alpha=0.5, edgecolors='k', linewidths=0.3)
lo = min(val_df[TARGET_COL].min(), val_pred.min()) * 0.9
hi = max(val_df[TARGET_COL].max(), val_pred.max()) * 1.05
axes[0].plot([lo, hi], [lo, hi], 'r--', label='Perfect prediction')
axes[0].set_xlabel('Predicted SalePrice ($)')
axes[0].set_ylabel('Actual SalePrice ($)')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()

# Residuals
residuals = val_df[TARGET_COL].values - val_pred
axes[1].scatter(val_pred, residuals, alpha=0.5, edgecolors='k', linewidths=0.3)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted SalePrice ($)')
axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Residual Plot')

fig.tight_layout()
plt.show()

## 8. Baseline Comparison

A `DummyRegressor` predicts the training mean for every house. Our model should beat it.

In [ ]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy='mean')
dummy.fit(train_df[TARGET_COL].values.reshape(-1, 1), train_df[TARGET_COL].values)
dummy_pred = dummy.predict(val_df[TARGET_COL].values.reshape(-1, 1))
dummy_metrics = evaluate(val_df[TARGET_COL].values, dummy_pred)

print('--- Dummy Baseline ---')
print(f'  MAE : ${dummy_metrics["MAE"]:,.2f}')
print(f'  RMSE: ${dummy_metrics["RMSE"]:,.2f}')
print(f'  R²  : {dummy_metrics["R2"]:.4f}')
print()
print('Our linear regression should have lower MAE/RMSE and higher R².')

## 9. Prediction on New Data

Use the fitted pipeline to predict a price for a hypothetical house.

In [ ]:
new_house = pd.DataFrame([{
    'GrLivArea': 1800,
    'BedroomAbvGr': 3,
    'FullBath': 2,
    'HalfBath': 1,
    'BsmtFullBath': 1,
    'BsmtHalfBath': 0,
}])

predicted_price = pipeline.predict(new_house)[0]
print(f'Predicted price for a 1800 sq ft, 3-bed, 2.5-bath house: ${predicted_price:,.0f}')

## 10. Key Takeaways

1. **Multiple linear regression** is a simple but useful baseline.
2. With only three features (area, bedrooms, bathrooms), we capture some signal but miss location, condition, age, etc.
3. Always evaluate on a held-out validation set — never on training data.
4. Compare against a baseline (DummyRegressor) to confirm the model adds value.
5. The Pipeline keeps preprocessing and prediction together, preventing data leakage.